# Cross-project prediction of Java code smells (SmellyCode++)

This notebook reproduces the experiment in the paper *A Study on Cross-Project Prediction of Java Code Smells Using Classical Machine-Learning Models*.

1. Run the first cell and upload `smellycode-xproject.zip`, or upload the folder to the Files pane.
2. Run the cells in order. A CPU runtime is enough, and the whole run takes about 45 minutes, most of it step 3.

Your outputs go to `work/`, and the outputs of the reported run are in `results/` for comparison.

In [ ]:
import os, zipfile
if not os.path.isdir('smellycode-xproject') and not os.path.exists('run_all.sh'):
    from google.colab import files
    up = files.upload()                      # choose smellycode-xproject.zip
    zipfile.ZipFile(next(iter(up))).extractall('.')
if os.path.isdir('smellycode-xproject'):
    os.chdir('smellycode-xproject')
print(os.getcwd(), sorted(os.listdir('.')))

## Install the pinned library versions

These are the versions used for the reported run. If a pin cannot be installed on Colab's Python version, install the same packages without the version numbers. The results may then differ slightly.

In [ ]:
!pip install -q -r requirements.txt

## Download SmellyCode++ (618 MB, CC0 1.0)

If the download fails, download `multi-smell-dataset-v1_2.csv` yourself from https://doi.org/10.6084/m9.figshare.28519385 and put it in `data/`.

In [ ]:
import hashlib, urllib.request
os.makedirs('data', exist_ok=True)
CSV = 'data/multi-smell-dataset-v1_2.csv'
if not os.path.exists(CSV):
    urllib.request.urlretrieve('https://ndownloader.figshare.com/files/52714583', CSV)
h = hashlib.sha256()
with open(CSV, 'rb') as f:
    for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
print(os.path.getsize(CSV), 'bytes', h.hexdigest())
assert h.hexdigest() == '971a81da3f55326a5a56bae59c3cde34c4ebab1a99615ac0c474ce5ecf5029d2', 'not the file used in the paper'

## Run the pipeline

Steps 1–2 clean the data and make the seeded project-level split. Step 3 tunes on the training projects only. Step 4 scores the frozen models once on the test projects.

In [ ]:
!python src/step1_clean_group.py data/multi-smell-dataset-v1_2.csv work
!python src/step2_split.py work
!python src/clusters.py work | tail -12

In [ ]:
!python src/step3_tune.py work

In [ ]:
!python src/step4_test.py work > work/step4.log && tail -1 work/step4.log
!python src/step5_contrast.py work data/multi-smell-dataset-v1_2.csv
!python src/step6_facts.py data/multi-smell-dataset-v1_2.csv work
!python src/fig3.py work/results.json work/fig3.png

## Key results

The next cell prints the test results per smell (Tables III and IV) and compares your run with the reported one.

In [ ]:
import json, pandas as pd
R = json.load(open('work/results.json')); ref = json.load(open('results/results.json'))
rows = []
for m in ['RF', 'LR', 'XGB']:
    for s, v in R['test'][m]['per_smell'].items():
        rows.append({'model': m, 'smell': s, **{k: round(x, 3) for k, x in v.items()}})
    rows.append({'model': m, 'smell': 'macro', 'F1': round(R['test'][m]['macroF1'], 3), 'PRAUC': round(R['test'][m]['macroAP'], 3)})
print('best model by validation:', R['best_model_by_validation'])
display(pd.DataFrame(rows))
same = all(abs(R['test'][m]['macroF1'] - ref['test'][m]['macroF1']) < 1e-9 for m in ['RF', 'LR', 'XGB'])
print('macro-F1 identical to the reported run:', same)

In [ ]:
from IPython.display import Image
Image('work/fig3.png', width=520)